In [70]:
import xgi
from hydra.core.data.utils import hypergraph_laplacian_zhou
from scipy.sparse.linalg import eigsh
from gdown import download
import tarfile
from os import remove, makedirs
from shutil import rmtree

In [71]:
gdrive_id = "1tpDiP1c73O18gCYEx4OI7kx8V_IdYxLt"

In [72]:
file_name = download(id=gdrive_id)

Downloading...
From: https://drive.google.com/uc?id=1tpDiP1c73O18gCYEx4OI7kx8V_IdYxLt
To: /home/valerio/Documents/hydra/examples/NDC-classes.tar.gz
100%|██████████| 421k/421k [00:00<00:00, 2.66MB/s]


In [73]:
dataset_name = file_name.split(".")[0]

In [74]:
with tarfile.open(file_name) as tar:
    tar.extractall()
remove(file_name)

/tmp/ipykernel_176539/2642023599.py:2: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


In [75]:
hyperedges = []
with open(f"{dataset_name}/{dataset_name}-nverts.txt") as f:
    with open(f"{dataset_name}/{dataset_name}-simplices.txt") as g:
        for nvert_line in f:
            nvert = int(nvert_line)
            # Read as many lines as nvert from g
            verts = []
            for _ in range(nvert):
                simplex_line = g.readline()
                simplex = int(simplex_line)
                verts.append(simplex)
            hyperedges.append(sorted(verts))
hyperedges = list(set(map(tuple, hyperedges)))
hypergraph = xgi.Hypergraph(hyperedges)

vertices_idx = sorted(hypergraph.nodes)
if vertices_idx != list(range(len(vertices_idx))):
    hyperedges = [[vertices_idx.index(v) for v in he] for he in hyperedges]
    hyperedges = list(set(map(tuple, hyperedges)))
hypergraph = xgi.Hypergraph(hyperedges)

In [76]:
incidence_matrix, coldict, rowdict = xgi.incidence.incidence_matrix(hypergraph, index=True)
Delta = hypergraph_laplacian_zhou(incidence_matrix) # TODO: Upload the node features directly on the repo
_, X = eigsh(Delta, k=128, which="SM")  # smallest magnitude eigenvalues
Delta = hypergraph_laplacian_zhou(incidence_matrix.T) # TODO: Upload the node features directly on the repo
_, Y = eigsh(Delta, k=128, which="SM")  # smallest magnitude eigenvalues
rowlist = [value for value in rowdict.values()]
collist = [value for value in coldict.values()]

In [77]:
for node in hypergraph.nodes:
    node_idx = collist.index(node)
    hypergraph.set_node_attributes({
        node: {"eigsh": X[node_idx].tolist()},
    })
for edge in hypergraph.edges:
    edge_idx = rowlist.index(edge)
    hypergraph.set_edge_attributes({
        edge: {"eigsh": Y[edge_idx].tolist()},
    })

In [78]:
hif = dict(xgi.to_hif_dict(hypergraph))

In [79]:
import json
rmtree(dataset_name)

makedirs(dataset_name + "/data", exist_ok=True)
with open(f"{dataset_name}/data/full-00000-of-00001.json", "w") as f:
    json.dump(hif, f)